# Stage 5 — SNOMED CT Entity Mapping

Map extracted clinical entities (symptoms, diagnoses mentioned, procedures, meds, labs)
from `patient_records/` to **offline SNOMED CT** (RF2 Snapshot under `data/SnomedCT_*`).

No cloud API — pure RF2 lexical match + fuzzy token/n-gram scoring.

**Input:** Stage 4 export  
**Output:** `data/stage_05_snomed_mapping/snomed_mappings.json`  
**Next:** `stage_06_snomed_ancestors.ipynb`

In [1]:
import sys
from pathlib import Path

NB = Path.cwd()
if not (NB / "pipeline.py").exists():
    NB = NB.parent if (NB.parent / "notebooks" / "pipeline.py").exists() else NB
    if (NB / "notebooks" / "pipeline.py").exists():
        NB = NB / "notebooks"
sys.path.insert(0, str(NB))
REPO = NB.parent if NB.name == "notebooks" else NB

from pipeline import EXPORT_DIR, print_pipeline_banner
from snomed_ct import (
    build_snomed_index,
    collect_entities_from_patient_records,
    export_mappings_to_patient_folders,
    find_snomed_root,
    run_stage05_mapping,
    write_json,
)

print_pipeline_banner()
SNOMED_ROOT = find_snomed_root(REPO / "data")
STAGE_05_DIR = REPO / "data" / "stage_05_snomed_mapping"
STAGE_05_DIR.mkdir(parents=True, exist_ok=True)
print(f"SNOMED root : {SNOMED_ROOT}")
print(f"Export dir  : {EXPORT_DIR}")
print(f"Stage 5 out : {STAGE_05_DIR}")

Pipeline mode : FULL (15 patients)
LLM provider  : OpenRouter (qwen/qwen-2.5-7b-instruct, ZDR on)
Qwen pair     : Local equivalent: ollama pull qwen2.5:7b
Admissions/patient (min): 2
Data dir      : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data
Export dir    : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
OpenRouter ZDR : enabled (provider.zdr=true on every request)
SNOMED root : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20250901T120000Z
Export dir  : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
Stage 5 out : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_05_snomed_mapping


/Users/narenkhatwani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
index = build_snomed_index(
    snomed_root=SNOMED_ROOT,
    cache_path=REPO / "data" / "snomed_index" / "snomed_index.pkl",
    force_rebuild=False,
)
print(f"Active concepts: {len(index.active_concepts):,}")

Loading SNOMED index cache → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/snomed_index/snomed_index.pkl
Active concepts: 382,170


In [3]:
entities = collect_entities_from_patient_records(EXPORT_DIR)
print(f"Entities: {len(entities)} | admissions: {len({(e['patient_id'], e['hadm_id']) for e in entities})}")

payload = run_stage05_mapping(entities, index)
print(f"Mapped: {payload['n_mapped']} | Unmapped: {payload['n_unmapped']}")

for row in payload["results"][:10]:
    s = row["snomed"]
    print(
        f"  [{row['kind']}] {row['term']!r}\n"
        f"    → {s.get('preferred_term')} | {s.get('concept_id')} | {s.get('match_method')} | score={s.get('score')}"
    )

Entities: 626 | admissions: 15
Mapped: 509 | Unmapped: 117
  [symptom] 'heavy vaginal bleeding'
    → Heavy episode of vaginal bleeding | 315224006 | fuzzy | score=0.7286607049870562
  [symptom] 'abdominal bloatedness'
    → None | None | None | score=0.0
  [symptom] 'vaginal spotting to light bleeding'
    → None | None | None | score=0.0
  [symptom] 'menorrhagia'
    → Menorrhagia | 386692008 | exact | score=1.0
  [symptom] 'dysmenorrhea'
    → Period pain | 266599000 | exact | score=1.0
  [symptom] 'intra-menstrual bleeding'
    → Intermenstrual bleeding | 237130006 | fuzzy | score=0.6247815373214463
  [symptom] 'post-coital bleeding'
    → Postcoital bleeding | 48880000 | fuzzy | score=0.6596448795998096
  [symptom] 'shortness of breath'
    → Dyspnea | 267036007 | exact | score=1.0
  [symptom] 'dyspareunia'
    → Dyspareunia | 71315007 | exact | score=1.0
  [diagnosis] 'abnormal uterine and vaginal bleeding'
    → Abnormal vaginal bleeding | 301822002 | fuzzy | score=0.74067848922

In [4]:
out = write_json(STAGE_05_DIR / "snomed_mappings.json", payload)
n = export_mappings_to_patient_folders(payload, EXPORT_DIR, "snomed_mapping.json")
print(f"Saved → {out}")
print(f"Per-admission files: {n}")
print("Next → stage_06_snomed_ancestors.ipynb")

Saved → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_05_snomed_mapping/snomed_mappings.json
Per-admission files: 15
Next → stage_06_snomed_ancestors.ipynb
